In [48]:
import pandas as pd
from typing import Dict

from freeze_thaw.data_preparation.general import align_timestamps_then_label, collect_cleaned_data
from freeze_thaw.config import config as c
from freeze_thaw.config import StationName
from freeze_thaw.data_preparation.splitting import train_test_split
from freeze_thaw.modeling.lgb_train import train_model
from freeze_thaw.data_preparation.feature_engineering import prepare_df
from freeze_thaw.modeling.lgb_test import lgb_pred

# Model Development

The goal is to train a lightGBM model and determine whether to use lagged features or not based on a time series cross-validation on 80% of the data.

---

## Table of Contents

1. **Setup**
    - *1.1 Variables*
    - *1.2 Functions*
2. **Training and Evaluation**
3. **Results**
    - 3.1 Without Lagged Features
    - 3.2 With Lagged Features
4. **Statistical Analysis**

---

## 1. Setup

### 1.1 Variables

In [49]:
lags = [1, 2, 3, 4, 5]
label_map = {
    c.CLASSES[0]: 0,
    c.CLASSES[1]: 1,
    c.CLASSES[2]: 2,
}
train_size = 0.8
n_splits = 5

### 1.2 Functions

In [50]:
def pipeline(stations: type[StationName],
             label_map: Dict[str, int],
             n_splits: int,
             train_size: float,
             lagged_features: bool,
             lags: list[int] | None = None) -> (pd.DataFrame, pd.DataFrame):
    """
    Train lgbm models for all stations using time series cross-validation then predict on test, with results
    for the train and test sets collected in separate dfs.
    :param stations: StationName class from config.py
    :param label_map: mapping of c.CLASSES to int labels
    :param n_splits: number of folds for cross-validation
    :param train_size: decimal fraction size of training data
    :param lagged_features: create lagged features or not
    :param lags: list of lags e.g. [1, 3] will create features for the backscatter from one and three datapoints prior
    :return: result_train_df and result_test_df
    """
    result_train_df = pd.DataFrame(columns=["Station", "Average Macro F1", "Average Transition F1"])
    result_test_df = pd.DataFrame(columns=["Station", "Macro F1", "Transition F1"])

    for station in stations:
        # create train/test split
        dfs = collect_cleaned_data(station, c.CLEANED_DATA_PATH, c.DATETIMEINDEX_NAME)
        ascat_df, _ = align_timestamps_then_label(dfs, c.ISMN_LONG_VAR_NAME, c.ASCAT_KEY_COLS, c.ERA5_KEY_COLS)
        ascat_df = ascat_df.drop(columns=c.ISMN_LONG_VAR_NAME)
        train, test = train_test_split(ascat_df, train_size)

        # train model
        train_prepared = prepare_df(train, label_encoding=label_map, lagged_features=lagged_features, lags=lags)
        train_result = train_model(train_prepared, n_splits=n_splits, label_encoding=label_map)

        # log train results
        result_train_df.loc[len(result_train_df)] = [station, train_result.average_macro_f1, train_result.average_transition_f1]

        # predict on test
        test_prepared = prepare_df(test, label_encoding=label_map, lagged_features=lagged_features, lags=lags)
        test_result = lgb_pred(test_prepared, model=train_result.model, label_encoding=label_map)

        # log test results
        result_test_df.loc[len(result_test_df)] = [station, test_result.macro_f1, test_result.transition_f1]

    return result_train_df, result_test_df

## 2. Training and Evaluation

Without lagged features

In [51]:
result_unlagged_train_df, result_unlagged_test_df = pipeline(StationName, label_map, n_splits, train_size, False)

Dropping 243 rows with no class label.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000439 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 758
[LightGBM] [Info] Number of data points in the train set: 2129, number of used features: 6
[LightGBM] [Info] Start training from score -1.554160
[LightGBM] [Info] Start training from score -1.799776
[LightGBM] [Info] Start training from score -0.472732
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000250 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 759
[LightGBM] [Info] Number of data points in the train set: 4253, number of used features: 6
[LightGBM] [Info] Start training from score -1.572055
[LightGBM] [Info] Start training from score -1.739315
[LightGBM] [Info] Start training from score 

With lagged features

In [55]:
result_lagged_train_df, result_lagged_test_df = pipeline(StationName, label_map, n_splits, train_size, True, lags)

Dropping 243 rows with no class label.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000303 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2134
[LightGBM] [Info] Number of data points in the train set: 2124, number of used features: 16
[LightGBM] [Info] Start training from score -1.551809
[LightGBM] [Info] Start training from score -1.797425
[LightGBM] [Info] Start training from score -0.474155
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000297 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2153
[LightGBM] [Info] Number of data points in the train set: 4248, number of used features: 16
[LightGBM] [Info] Start training from score -1.570878
[LightGBM] [Info] Start training from score -1.738138
[LightGBM] [Info] Start training from sc

## 3. Results

### 3.1 Without Lagged Features

#### Train

In [52]:
result_unlagged_train_df

,Station,Average Macro F1,Average Transition F1
0,Aberdeen-35-WNW,0.654033,0.494805
1,Jamestown-38-WSW,0.710539,0.497018
2,GobblersKnob,0.827562,0.607234
3,Nenana,0.772453,0.533890
4,L23,0.876436,0.737274
5,L38,0.782841,0.537092
6,NST-07,0.860428,0.713538
7,NST-09,0.830526,0.627171
8,SOD012,0.682619,0.673759
9,SOD103,0.583262,0.848254


#### Test

In [58]:
result_unlagged_test_df

,Station,Macro F1,Transition F1
0,Aberdeen-35-WNW,0.644785,0.571429
1,Jamestown-38-WSW,0.705044,0.645207
2,GobblersKnob,0.838089,0.660377
3,Nenana,0.793644,0.645978
4,L23,0.871133,0.715953
5,L38,0.794368,0.560976
6,NST-07,0.735672,0.528529
7,NST-09,0.819869,0.633776
8,SOD012,0.676954,0.834813
9,SOD103,0.558348,0.784304


### 3.2 With Lagged Features

#### Train

In [56]:
result_lagged_train_df

,Station,Average Macro F1,Average Transition F1
0,Aberdeen-35-WNW,0.643188,0.500146
1,Jamestown-38-WSW,0.686354,0.468523
2,GobblersKnob,0.837881,0.634682
3,Nenana,0.792990,0.584745
4,L23,0.869935,0.722601
5,L38,0.761769,0.481633
6,NST-07,0.838333,0.661454
7,NST-09,0.814863,0.590362
8,SOD012,0.688241,0.703760
9,SOD103,0.583583,0.857651


#### Test

In [57]:
result_lagged_test_df

,Station,Macro F1,Transition F1
0,Aberdeen-35-WNW,0.649312,0.586942
1,Jamestown-38-WSW,0.729882,0.671121
2,GobblersKnob,0.844992,0.675296
3,Nenana,0.788663,0.640871
4,L23,0.881123,0.750903
5,L38,0.806479,0.596078
6,NST-07,0.734136,0.515723
7,NST-09,0.819458,0.630769
8,SOD012,0.672572,0.826714
9,SOD103,0.582475,0.835792


### 3.3 Difference in Scores

Show the scores for lagged - unlagged, with positive scores indicating better performance with lagged features.

#### Train

In [59]:
train_diff = result_lagged_train_df.copy()
float_cols = ["Average Macro F1", "Average Transition F1"]
train_diff[float_cols] = result_lagged_train_df[float_cols] - result_unlagged_train_df[float_cols]
train_diff = train_diff.rename(columns={
    "Average Macro F1": "Average Macro F1 Difference",
    "Average Transition F1": "Average Transition F1 Difference",
})

train_diff

,Station,Average Macro F1 Difference,Average Transition F1 Difference
0,Aberdeen-35-WNW,-0.010846,0.005341
1,Jamestown-38-WSW,-0.024185,-0.028495
2,GobblersKnob,0.010319,0.027447
3,Nenana,0.020538,0.050855
4,L23,-0.006500,-0.014673
5,L38,-0.021073,-0.055459
6,NST-07,-0.022095,-0.052083
7,NST-09,-0.015663,-0.036809
8,SOD012,0.005621,0.030001
9,SOD103,0.000321,0.009397


#### Test

In [60]:
test_diff = result_lagged_test_df.copy()
float_cols = ["Macro F1", "Transition F1"]
test_diff[float_cols] = result_lagged_test_df[float_cols] - result_unlagged_test_df[float_cols]
test_diff = test_diff.rename(columns={
    "Macro F1": "Macro F1 Difference",
    "Transition F1": "Transition F1 Difference",
})

test_diff

,Station,Macro F1 Difference,Transition F1 Difference
0,Aberdeen-35-WNW,0.004527,0.015513
1,Jamestown-38-WSW,0.024838,0.025914
2,GobblersKnob,0.006903,0.014918
3,Nenana,-0.004981,-0.005107
4,L23,0.009990,0.034949
5,L38,0.012111,0.035103
6,NST-07,-0.001535,-0.012805
7,NST-09,-0.000411,-0.003007
8,SOD012,-0.004383,-0.008099
9,SOD103,0.024127,0.051489


## 4. Statistical Analysis